### 0) Imports

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from category_encoders import CountEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
import catboost as cb
import lightgbm as lgb

### 1) Download data from Don’tGetKicked competition. Design train/validation/test split and preprocess categorical variables

In [ ]:
df = pd.read_csv('../datasets/training.csv')

In [3]:
def split_test_valid_train(data, date_col):
    df = data.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df.sort_values(by=date_col, ignore_index=True, inplace=True)

    uniq_dates = sorted(df[date_col].unique())
    n = len(uniq_dates)
    train_end = uniq_dates[n // 3]
    valid_end = uniq_dates[2 * n // 3]

    train = df[df[date_col] < train_end].copy()
    valid = df[(df[date_col] >= train_end) & (df[date_col] < valid_end)].copy()
    test  = df[df[date_col] >= valid_end].copy()

    return train, valid, test

In [4]:
X_train, X_valid, X_test = split_test_valid_train(df, 'PurchDate')

In [5]:
df.head(10)

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,12/7/2009,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,12/7/2009,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,...,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,12/7/2009,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,...,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020
5,6,0,12/7/2009,ADESA,2004,5,MITSUBISHI,GALANT 4C,ES,4D SEDAN ES,...,8149.0,9451.0,NaN,NaN,19638,33619,FL,5600.0,0,594
6,7,0,12/7/2009,ADESA,2004,5,KIA,SPECTRA,EX,4D SEDAN EX,...,6230.0,8603.0,NaN,NaN,19638,33619,FL,4200.0,0,533
7,8,0,12/7/2009,ADESA,2005,4,FORD,TAURUS,SE,4D SEDAN SE,...,6942.0,8242.0,NaN,NaN,19638,33619,FL,4500.0,0,825
8,9,0,12/7/2009,ADESA,2007,2,KIA,SPECTRA,EX,4D SEDAN EX,...,9637.0,10778.0,NaN,NaN,21973,33619,FL,5600.0,0,482
9,10,0,12/7/2009,ADESA,2007,2,FORD,FIVE HUNDRED,SEL,4D SEDAN SEL,...,12580.0,14845.0,NaN,NaN,21973,33619,FL,7700.0,0,1633


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              72983 non-null  int64  
 1   IsBadBuy                           72983 non-null  int64  
 2   PurchDate                          72983 non-null  object 
 3   Auction                            72983 non-null  object 
 4   VehYear                            72983 non-null  int64  
 5   VehicleAge                         72983 non-null  int64  
 6   Make                               72983 non-null  object 
 7   Model                              72983 non-null  object 
 8   Trim                               70623 non-null  object 
 9   SubModel                           72975 non-null  object 
 10  Color                              72975 non-null  object 
 11  Transmission                       72974 non-null  obj

In [7]:
y_train, y_valid, y_test = X_train['IsBadBuy'], X_valid['IsBadBuy'], X_test['IsBadBuy']

In [8]:
drop_cols = ['IsBadBuy', 'PurchDate', 'RefID', 'KickDate', 'BYRNO', 'WheelTypeID']
ohe_cols = ['Auction', 'Transmission', 'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART']
count_cols = ['Make', 'Model', 'Trim', 'SubModel', 'Color', 'VNST']
num_cols = df.select_dtypes('number').columns.tolist()
num_cols = [num for num in num_cols if num not in drop_cols]

In [9]:
imputer = SimpleImputer(strategy='median')

In [10]:
ohe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))])

In [11]:
count = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('count', CountEncoder(handle_unknown=0))])

In [12]:
preprocess = ColumnTransformer(
    transformers=[
        ('num', imputer, num_cols),
        ('ohe', ohe, ohe_cols),
        ('count', count, count_cols),
    ],
    remainder='drop'
)

In [13]:
X_train = preprocess.fit_transform(X_train)
X_valid = preprocess.transform(X_valid)
X_test = preprocess.transform(X_test)

In [14]:
def Gini(y_true, y_pred):
    return np.abs(2 * roc_auc_score(y_true, y_pred) - 1)

### 2) Create a Python class for Decision Tree Classifier and Decision Tree Regressor (MSE loss)

In [15]:
class DecisionNode:
    def __init__(self, feature_i=None, threshold=None,
                 value=None, probs=None, true_branch=None, false_branch=None):
        self.feature_i = feature_i
        self.threshold = threshold
        self.value = value
        self.probs = probs
        self.true_branch = true_branch
        self.false_branch = false_branch

    def _calculate_gini(self, y):
        counts = np.bincount(y.reshape(-1).astype(int))
        probs = counts / len(y)
        return 1 - np.sum(probs**2)

    def _calculate_std(self, y):
        return np.std(y)

In [16]:
class DecisionTree:
    def __init__(self, min_samples_split=2, min_impurity=1e-7,
                 max_depth=float('inf'), loss=None):
        self.root = None
        self.min_samples_split = min_samples_split
        self.min_impurity = min_impurity
        self.max_depth = max_depth
        self._impurity_calculation = None
        self._leaf_value_calculation = None
        self._leaf_probs_calculation = None
        self.loss = loss

    def fit (self, X, y, loss=None):
        self.root = self._build_tree(X, y)
        self.loss = None

    def _build_tree(self, X, y, current_depth=0):
        largest_impurity = 0
        best_criteria = None
        best_sets = None

        if len(np.shape(y)) == 1:
            y = np.expand_dims(y, axis=1)

        Xy = np.concatenate((X, y), axis=1)

        n_samples, n_features = np.shape(X)

        if n_samples >= self.min_samples_split and current_depth <= self.max_depth:
            for feature_i in range(n_features):
                unique_values = np.unique(X[:, feature_i])
                for threshold in unique_values:
                    Xy1, Xy2 = self._divide_on_feature(Xy, feature_i, threshold)

                    if len(Xy1) > 0 and len(Xy2) > 0:
                        y1 = Xy1[:, n_features:]
                        y2 = Xy2[:, n_features:]

                        impurity = self._impurity_calculation(y, y1, y2)
                        if impurity > largest_impurity:
                            largest_impurity = impurity
                            best_criteria = {'feature_i':feature_i, 'threshold':threshold}
                            best_sets = {
                                'leftX':Xy1[:, :n_features],
                                'lefty':Xy1[:, n_features:],
                                'rightX':Xy2[:, :n_features],
                                'righty':Xy2[:, n_features:]
                            }

        if largest_impurity > self.min_impurity:
            true_branch = self._build_tree(best_sets['leftX'], best_sets['lefty'], current_depth + 1)
            false_branch = self._build_tree(best_sets['rightX'], best_sets['righty'], current_depth + 1)
            return DecisionNode(feature_i=best_criteria['feature_i'], threshold=best_criteria['threshold'],
                                true_branch=true_branch, false_branch=false_branch)

        leaf_value = self._leaf_value_calculation(y)
        leaf_probs = self._leaf_probs_calculation(y)

        return DecisionNode(value=leaf_value, probs=leaf_probs)

    def _divide_on_feature(self, X, feature_i, threshold):
        idx_true = X[:, feature_i] >= threshold
        Xy1 = X[idx_true]
        Xy2 = X[~idx_true]
        return Xy1, Xy2

    def predict_value(self, x, tree=None):
        if tree is None:
            tree = self.root

        if tree.value is not None:
            return tree.value

        feature_value = x[tree.feature_i]
        branch = tree.false_branch
        if feature_value >= tree.threshold:
            branch = tree.true_branch

        return self.predict_value(x, branch)

    def predict(self, X):
        y_pred = np.array([self.predict_value(sample) for sample in X])
        return y_pred

In [17]:
class MyDecisionTreeClassifier(DecisionTree):
    def _calculate_information_gain(self, y, y1, y2):
        node = DecisionNode()
        parent_gini = node._calculate_gini(y)
        p = len(y1) / len(y)
        child_gini = p * node._calculate_gini(y1) + (1 - p) * node._calculate_gini(y2)
        return parent_gini - child_gini

    def _calculate_probs(self, y):
        counts = np.bincount(y.reshape(-1).astype(int), minlength=2)
        return counts / len(y)

    def _majority_vote(self, y):
        most_common = None
        max_count = 0
        for label in np.unique(y):
            count = len(y[y == label])
            if count > max_count:
                most_common = label
                max_count = count
        return most_common

    def predict_probs(self, x, tree=None):
        if tree is None:
            tree = self.root

        if tree.probs is not None:
            return tree.probs

        feature_probs = x[tree.feature_i]
        branch = tree.false_branch
        if feature_probs >= tree.threshold:
            branch = tree.true_branch

        return self.predict_probs(x, branch)

    def predict_proba(self, X):
        y_pred = np.array([self.predict_probs(sample) for sample in X])
        return y_pred

    def fit(self, X, y):
        self._impurity_calculation = self._calculate_information_gain
        self._leaf_value_calculation = self._majority_vote
        self._leaf_probs_calculation = self._calculate_probs
        super(MyDecisionTreeClassifier, self).fit(X, y)

In [18]:
class MyDecisionTreeRegressor(DecisionTree):
    def _calculate_std(self, y, y1, y2):
        node = DecisionNode()
        std = node._calculate_std(y)
        p = len(y1) / len(y)
        std_value = std - (p * node._calculate_std(y1) + (1 - p) * node._calculate_std(y2))
        return std_value

    def fit(self, X, y):
        self._impurity_calculation = self._calculate_std
        self._leaf_value_calculation = lambda y: np.mean(y)
        self._leaf_probs_calculation = lambda y: None
        super(MyDecisionTreeRegressor, self).fit(X, y)

### 3) Use MyDecisionTreeClassifier and check its performance on the validation dataset

In [19]:
mydtc = MyDecisionTreeClassifier(max_depth=7)
mydtc.fit(X_train, y_train)
y_pred = mydtc.predict_proba(X_valid)[:,1]

In [20]:
print(f'Gini score for MyDecisionTreeClassifier = {Gini(y_valid, y_pred)}')

Gini score for MyDecisionTreeClassifier = 0.28687307421934416


### 4) Use sklearn's DecisionTreeClassifier and check its performance on the validation dataset

In [21]:
dtc = DecisionTreeClassifier(max_depth=7, min_impurity_decrease=1e-7)
dtc.fit(X_train, y_train)
y_pred = dtc.predict_proba(X_valid)[:,1]

In [22]:
print(f'Gini score for DecisionTreeClassifier = {Gini(y_valid, y_pred)}')

Gini score for DecisionTreeClassifier = 0.2886474621157922


### 5) Implement the RandomForestClassifier and check its performance

In [23]:
class MyRandomForestClassifier:
    def __init__(self, n_estimators=100, max_features=None, min_samples_split=2,
                max_depth=float('inf'), min_impurity=1e-7, seed=21):
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.min_impurity = min_impurity
        self.seed = seed
        self.trees = []
        for _ in range(n_estimators):
            self.trees.append(
                MyDecisionTreeClassifier(
                    min_samples_split = self.min_samples_split,
                    max_depth = self.max_depth,
                    min_impurity = self.min_impurity
                )
            )

    def fit(self, X, y):
        np.random.seed(self.seed)
        n_samples, n_features = X.shape

        if not self.max_features:
            self.max_features = int(np.sqrt(n_features))

        for i in range(self.n_estimators):
            idx_samples = np.random.choice(range(n_samples), size=n_samples, replace=True)
            X_subset = X[idx_samples]
            y_subset = y[idx_samples]

            idx_features = np.random.choice(range(n_features), size=self.max_features, replace=True)
            self.trees[i].feature_indices = idx_features
            self.trees[i].fit(X_subset[:, idx_features], y_subset)

    def predict_proba(self, X):
        tree_probs = []
        for tree in self.trees:
            X_subset = X[:, tree.feature_indices]
            probs = np.array(tree.predict_proba(X_subset))
            tree_probs.append(probs)
        return np.mean(tree_probs, axis=0)

    def predict(self, X):
        y_pred = self.predict_proba(X)
        return np.argmax(y_pred, axis=1)

In [24]:
myrfc = MyRandomForestClassifier(n_estimators=10, max_depth=7)
myrfc.fit(X_train, y_train)
y_pred = myrfc.predict_proba(X_valid)[:,1]

In [25]:
print(f'Gini score for MyRandomForestClassifier = {Gini(y_valid, y_pred)}')

Gini score for MyRandomForestClassifier = 0.2854490737095592


### 6) Use your DecisionTree design class for GBDT classifier

In [26]:
class MyGBDTClassifier:
    def __init__(self, n_estimators=200, learning_rate=0.5, min_samples_split=2,
                 min_impurity=1e-7, max_depth=float('inf'), max_features=None):
        self.n_estimators = n_estimators
        self.learning_rate=0.5
        self.min_samples_split = min_samples_split
        self.min_impurity = min_impurity
        self.max_depth = max_depth
        self.max_features = max_features
        self.init_value = None
        self.trees = []

    def _sigmoid(self, a):
        return 1 / (1 + np.exp(-a))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.init_value = np.log(np.mean(y) / (1 - np.mean(y)))
        current_value = np.full(y.shape, self.init_value)

        if self.max_features is None:
            self.max_features = n_features // 3

        for i in range(self.n_estimators):
            probs = self._sigmoid(current_value)
            gradient = probs - y

            idx_features = np.random.choice(range(n_features), size=self.max_features, replace=True)
            tree = MyDecisionTreeRegressor(
                max_depth=self.max_depth
            )
            tree.fit(X[:, idx_features], gradient)
            tree.feature_indices_ = idx_features

            update = tree.predict(X[:, idx_features])
            current_value -= self.learning_rate * update

            self.trees.append(tree)

    def predict_proba(self, X):
        value = np.full(X.shape[0], self.init_value)
        for tree in self.trees:
            value -= self.learning_rate * tree.predict(X[:, tree.feature_indices_])

        prob1 = self._sigmoid(value)
        return np.column_stack((1 - prob1, prob1))

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

In [27]:
mygbdtc = MyGBDTClassifier(n_estimators=10, max_depth=7)
mygbdtc.fit(X_train, y_train)
y_pred = mygbdtc.predict_proba(X_valid)[:,1]

In [28]:
print(f'Gini score for MyGBDTClassifier = {Gini(y_valid, y_pred)}')

Gini score for MyGBDTClassifier = 0.3390986450657034


### 7) Use LightGBM, Catboost, and XGBoost for fitting on a training set and prediction on a validation set

In [29]:
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    booster='dart',
    max_depth=7
)
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict_proba(X_valid)[:,1]

In [30]:
print(f'Gini score for XGBClassifier = {Gini(y_valid, y_pred)}')

Gini score for XGBClassifier = 0.35059111832429846


In [31]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    force_col_wise=True
)
lgb_model.fit(X_train, y_train)
y_pred = lgb_model.predict_proba(X_valid)[:,1]

[LightGBM] [Info] Number of positive: 2607, number of negative: 20452
[LightGBM] [Info] Total Bins 3587
[LightGBM] [Info] Number of data points in the train set: 23059, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.113058 -> initscore=-2.059881
[LightGBM] [Info] Start training from score -2.059881


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [32]:
print(f'Gini score for LGBMClassifier = {Gini(y_valid, y_pred)}')

Gini score for LGBMClassifier = 0.3451946865217428


In [33]:
cb_model = cb.CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=7
)
cb_model.fit(X_train, y_train)
y_pred = cb_model.predict_proba(X_valid)[:,1]

0:	learn: 0.6110200	total: 70.4ms	remaining: 6.97s
1:	learn: 0.5464877	total: 93.8ms	remaining: 4.59s
2:	learn: 0.4981100	total: 109ms	remaining: 3.54s
3:	learn: 0.4628361	total: 124ms	remaining: 2.98s
4:	learn: 0.4339200	total: 139ms	remaining: 2.64s
5:	learn: 0.4125237	total: 153ms	remaining: 2.4s
6:	learn: 0.3949394	total: 168ms	remaining: 2.23s
7:	learn: 0.3804865	total: 183ms	remaining: 2.11s
8:	learn: 0.3706348	total: 198ms	remaining: 2s
9:	learn: 0.3618565	total: 214ms	remaining: 1.93s
10:	learn: 0.3551101	total: 229ms	remaining: 1.85s
11:	learn: 0.3487707	total: 245ms	remaining: 1.8s
12:	learn: 0.3447940	total: 260ms	remaining: 1.74s
13:	learn: 0.3414404	total: 279ms	remaining: 1.72s
14:	learn: 0.3380202	total: 299ms	remaining: 1.69s
15:	learn: 0.3349882	total: 316ms	remaining: 1.66s
16:	learn: 0.3329720	total: 331ms	remaining: 1.62s
17:	learn: 0.3315498	total: 346ms	remaining: 1.57s
18:	learn: 0.3293233	total: 362ms	remaining: 1.54s
19:	learn: 0.3275133	total: 378ms	remaining:

In [34]:
print(f'Gini score for CatBoostClassifier = {Gini(y_valid, y_pred)}')

Gini score for CatBoostClassifier = 0.3617229272417164


### 8) Take the best model and estimate its performance on the test dataset

В моем случае лучшая модель это CatBoostClassifier

In [35]:
cb_model = cb.CatBoostClassifier(
    iterations=100,
    learning_rate=0.01,
    depth=3
)
cb_model.fit(X_train, y_train)
y_pred = cb_model.predict_proba(X_train)[:,1]

0:	learn: 0.6847497	total: 6.71ms	remaining: 664ms
1:	learn: 0.6764483	total: 12.9ms	remaining: 634ms
2:	learn: 0.6685045	total: 19ms	remaining: 613ms
3:	learn: 0.6607827	total: 25.1ms	remaining: 602ms
4:	learn: 0.6531744	total: 31.6ms	remaining: 600ms
5:	learn: 0.6458941	total: 37.8ms	remaining: 592ms
6:	learn: 0.6388449	total: 43.4ms	remaining: 576ms
7:	learn: 0.6317252	total: 51.8ms	remaining: 596ms
8:	learn: 0.6245840	total: 58ms	remaining: 586ms
9:	learn: 0.6178893	total: 63.8ms	remaining: 574ms
10:	learn: 0.6112986	total: 70.4ms	remaining: 569ms
11:	learn: 0.6047893	total: 76.4ms	remaining: 560ms
12:	learn: 0.5985093	total: 82.4ms	remaining: 551ms
13:	learn: 0.5922134	total: 88.7ms	remaining: 545ms
14:	learn: 0.5862264	total: 95.2ms	remaining: 539ms
15:	learn: 0.5804128	total: 102ms	remaining: 534ms
16:	learn: 0.5747126	total: 108ms	remaining: 525ms
17:	learn: 0.5691754	total: 114ms	remaining: 521ms
18:	learn: 0.5636814	total: 121ms	remaining: 516ms
19:	learn: 0.5584159	total: 12

In [36]:
print(f'Gini score for CatBoostClassifier on training data = {Gini(y_train, y_pred)}')

Gini score for CatBoostClassifier on training data = 0.35866070459326194


In [37]:
cb_model.fit(X_train, y_train)
y_pred = cb_model.predict_proba(X_valid)[:,1]

0:	learn: 0.6847497	total: 7.22ms	remaining: 715ms
1:	learn: 0.6764483	total: 13.5ms	remaining: 663ms
2:	learn: 0.6685045	total: 19.6ms	remaining: 635ms
3:	learn: 0.6607827	total: 25.9ms	remaining: 622ms
4:	learn: 0.6531744	total: 32.6ms	remaining: 619ms
5:	learn: 0.6458941	total: 38.6ms	remaining: 605ms
6:	learn: 0.6388449	total: 44.4ms	remaining: 589ms
7:	learn: 0.6317252	total: 51.2ms	remaining: 588ms
8:	learn: 0.6245840	total: 57.1ms	remaining: 577ms
9:	learn: 0.6178893	total: 63.1ms	remaining: 568ms
10:	learn: 0.6112986	total: 69.7ms	remaining: 564ms
11:	learn: 0.6047893	total: 75.9ms	remaining: 556ms
12:	learn: 0.5985093	total: 82.1ms	remaining: 550ms
13:	learn: 0.5922134	total: 89ms	remaining: 547ms
14:	learn: 0.5862264	total: 95.7ms	remaining: 543ms
15:	learn: 0.5804128	total: 102ms	remaining: 535ms
16:	learn: 0.5747126	total: 108ms	remaining: 527ms
17:	learn: 0.5691754	total: 115ms	remaining: 522ms
18:	learn: 0.5636814	total: 121ms	remaining: 517ms
19:	learn: 0.5584159	total: 

In [38]:
print(f'Gini score for CatBoostClassifier on validation data = {Gini(y_valid, y_pred)}')

Gini score for CatBoostClassifier on validation data = 0.32440838089823054


In [39]:
cb_model.fit(X_train, y_train)
y_pred = cb_model.predict_proba(X_test)[:,1]

0:	learn: 0.6847497	total: 20.2ms	remaining: 2s
1:	learn: 0.6764483	total: 38.1ms	remaining: 1.87s
2:	learn: 0.6685045	total: 44.3ms	remaining: 1.43s
3:	learn: 0.6607827	total: 50.2ms	remaining: 1.21s
4:	learn: 0.6531744	total: 56.8ms	remaining: 1.08s
5:	learn: 0.6458941	total: 62.9ms	remaining: 986ms
6:	learn: 0.6388449	total: 68.6ms	remaining: 911ms
7:	learn: 0.6317252	total: 75.1ms	remaining: 863ms
8:	learn: 0.6245840	total: 81ms	remaining: 819ms
9:	learn: 0.6178893	total: 86.8ms	remaining: 781ms
10:	learn: 0.6112986	total: 93.4ms	remaining: 755ms
11:	learn: 0.6047893	total: 99.4ms	remaining: 729ms
12:	learn: 0.5985093	total: 106ms	remaining: 706ms
13:	learn: 0.5922134	total: 112ms	remaining: 686ms
14:	learn: 0.5862264	total: 118ms	remaining: 671ms
15:	learn: 0.5804128	total: 125ms	remaining: 654ms
16:	learn: 0.5747126	total: 131ms	remaining: 638ms
17:	learn: 0.5691754	total: 137ms	remaining: 625ms
18:	learn: 0.5636814	total: 144ms	remaining: 613ms
19:	learn: 0.5584159	total: 150ms	

In [40]:
print(f'Gini score for CatBoostClassifier on testing data = {Gini(y_test, y_pred)}')

Gini score for CatBoostClassifier on testing data = 0.3504206291342289


Видно, что разрыв между тренировочными, валидационными и тренировочными данными по Джини мал. Лучший результат на тренировочных данных, но это объясняется тем, что данные обучались именно на этих данных, поэтому моделе было проще выучить какие либо зависимости, а разница между результатами на тестовых и валидационных данных наоборот показывает, что модель не запомнила какие то частные случаи, а наоборот нашла общие зависимости, поэтому результат на тестовых лучше, чем на валидационных, а значит модель не переобучена